# Figure 5 — Analog generation

In [1]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D
from scipy.stats import kruskal
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
matplotlib.font_manager._load_fontmanager(try_read_cache=False)
plt.style.use('seaborn-v0_8-paper')
plt.rcParams.update({
    'font.size': 6, 'axes.titlesize': 6, 'axes.labelsize': 6,
    'xtick.labelsize': 6, 'ytick.labelsize': 6, 'legend.fontsize': 6,
    'axes.linewidth': 0.6, 'axes.edgecolor': '#333333',
    'xtick.color': '#333333', 'ytick.color': '#333333',
    'axes.labelcolor': '#222222', 'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Liberation Sans', 'DejaVu Sans'],
    'figure.dpi': 300, 'savefig.dpi': 600,
    'xtick.major.width': 0.5, 'ytick.major.width': 0.5,
    'xtick.major.size': 2.5, 'ytick.major.size': 2.5,
    'xtick.major.pad': 2, 'ytick.major.pad': 2,
    'axes.spines.top': False, 'axes.spines.right': False,
})

DATA = '/home/pszymczak/code/omegamp-dashboard/data/'

In [2]:
ref = pd.read_csv(DATA + 'omegamp_reference_table.csv')
mic = pd.read_csv(DATA + 'mic.csv')

hc50 = pd.read_csv(DATA + 'hc50.csv')
cc50 = pd.read_csv(DATA + 'cc50.csv')
_dc = lambda v: float(str(v).lstrip('>')) if pd.notna(v) else float('nan')
for _c in mic.columns[1:]:
    mic[_c] = mic[_c].apply(_dc)
hc50['HC50'] = hc50['HC50'].apply(_dc)
cc50['CC50'] = cc50['CC50'].apply(_dc)

npn_fc = pd.read_csv(DATA + 'npn_fc.csv')
disc_fc = pd.read_csv(DATA + 'disc_fc.csv')
npn = npn_fc[['short_name', 'MaxRel', 'AUC']].copy()
disc = disc_fc[['short_name', 'MaxRel', 'AUC']].copy()

# Fixed assay concentrations per family (μM), shown in panels F/G y-tick labels
FAM_CONC = {
    'As-CATH4-6L': 1.0, 'Mammutin-1': 16.0, 'GQ20': 4.0,
    'DeNo1047': 4.0, 'BoCo1': 2.0, 'OP-145-TII4': 32.0,
}

# Deduplicate mic.csv (Ω-MT-bZIP-4 appears twice with conflicting values;
# DNA-binding objective so no figure panel is currently affected)
dups = mic[mic['short_name'].duplicated(keep=False)]['short_name'].unique()
if len(dups):
    print(f"WARNING: duplicate short_names in mic.csv — keeping first row: {dups.tolist()}")
    mic = mic.drop_duplicates(subset='short_name', keep='first')

# MIC: NaN means not tested, treated as right-censored at the detection limit (> 64 μM)
mic_strains = mic.columns[1:]
mic_vals = mic[mic_strains].astype(float)
mic['n_strains_le2'] = (mic_vals <= 2).sum(axis=1)
mic['n_strains_le4'] = (mic_vals <= 4).sum(axis=1)
mic['mic50'] = mic_vals.fillna(64).clip(upper=64).median(axis=1)

# NPN and DiSC should always be co-measured; warn if not
npn_names = set(npn['short_name'])
disc_names = set(disc['short_name'])
if npn_names != disc_names:
    print(f"WARNING: NPN-only: {npn_names - disc_names}; DiSC-only: {disc_names - npn_names}")

# Build merged analysis frame
df = (ref
      .merge(hc50, on='short_name', how='left')
      .merge(cc50, on='short_name', how='left')
      .merge(npn.rename(columns={'MaxRel': 'NPN', 'AUC': 'NPN_AUC'}), on='short_name', how='left')
      .merge(disc.rename(columns={'MaxRel': 'DiSC', 'AUC': 'DiSC_AUC'}), on='short_name', how='left')
      .merge(mic[['short_name', 'n_strains_le2', 'n_strains_le4', 'mic50']], on='short_name', how='left')
)

# HC50/CC50: save raw values then clip to display range [0.1, 128] μM
df['HC50_raw'] = df['HC50']
df['CC50_raw'] = df['CC50']
df['HC50'] = df['HC50'].clip(0.1, 128)
df['CC50'] = df['CC50'].clip(0.1, 128)
# Safety window = HC50 / MIC50 (both μM); floor aligned with colorbar minimum
df['SW'] = (df['HC50'] / df['mic50']).clip(0.1, 128)

# Map internal prototype IDs to family names for two families loaded via DBAASPS IDs
PROTO_ID_MAP = {'DBAASPS_20015': 'As-CATH4-6L', 'DBAASPS_20955': 'OP-145-TII4'}
PROTOTYPE_FAMILY_NAMES = [
    'BoCo1', 'GQ20', 'Mammutin-1', 'DeNo1047',
    'cecropin', 'LG21', 'pa4', 'sarcotoxin', 'bZIP',
]

def get_family(row):
    category = row['category']
    if category == 'prototype':
        name = row['short_name']
        for fam in PROTOTYPE_FAMILY_NAMES:
            if fam.lower() in name.lower():
                return fam
        if 'OP-145' in name:
            return 'OP-145-TII4'
        if 'As-CATH' in name:
            return 'As-CATH4-6L'
        return name
    if category == 'analog':
        proto_id = row.get('prototype', None)
        if pd.isna(proto_id):
            return ''
        return PROTO_ID_MAP.get(str(proto_id), str(proto_id))
    return ''  # de_novo and other categories

df['family'] = df.apply(get_family, axis=1)
proto_by_fam = {row['family']: row for _, row in df[df['category'] == 'prototype'].iterrows()}
antimicrobial = df[(df['category'] == 'analog') & (df['objective'] == 'antimicrobial')].copy()



In [3]:
FC = {
    'Mammutin-1': '#D55E00', 'GQ20': '#E69F00', 'BoCo1': '#009E73',
    'DeNo1047': '#56B4E9', 'As-CATH4-6L': '#0072B2', 'OP-145-TII4': '#CC79A7',
}
INACTIVE_ORDER = ['Mammutin-1', 'GQ20', 'As-CATH4-6L', 'BoCo1', 'OP-145-TII4', 'DeNo1047']

LEADS_SHORT = {
    'Ω-AT-BoCo1-5': 'Ω-B-5', 'Ω-AT-BoCo1-9': 'Ω-B-9',
    'Ω-AU-GQ20-4': 'Ω-G-4', 'Ω-AP-Mammutin-1-4': 'Ω-M-4',
}
LEADS_SET = set(LEADS_SHORT.keys())

def annotate_lead(ax, x, y, label, offset=(12, 8), ha='left'):
    ax.annotate(label, (x, y), fontsize=6, ha=ha, va='center', color='black',
                xytext=offset, textcoords='offset points',
                arrowprops=dict(arrowstyle='-', color='#555', lw=0.5), zorder=10)

def _dplot_b(ax, sdf_g, show_xlabel=True):
    y = np.arange(len(sdf_g))
    ax.barh(y, sdf_g['moderate'], height=0.7, color='#CCCCCC', label='MIC $\\leq$ 32 $\\mu$M', zorder=1)
    ax.barh(y, sdf_g['active'],   height=0.7, color='#FFD580', label='MIC $\\leq$ 4 $\\mu$M',  zorder=2)
    ax.barh(y, sdf_g['potent'],   height=0.7, color='#E8912D', label='MIC $\\leq$ 2 $\\mu$M',  zorder=3)
    ax.set_yticks(np.arange(len(sdf_g)))
    ylbls = ax.set_yticklabels(sdf_g['name'], fontsize=6)
    for i, (_, row) in enumerate(sdf_g.iterrows()):
        ylbls[i].set_color('#DC2626' if row['gram'] == '+' else '#2563EB')
        if row['mdr']:
            ylbls[i].set_fontweight('bold')
    ax.set_xlim(0, 105)
    if show_xlabel:
        ax.set_xlabel('Success rate (%)', fontsize=6)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

STRAIN_LABELS = {
    'A. baumannii ATCC 19606 (-)':                   '$\\it{A. baumannii}$ ATCC 19606',
    'A. baumannii ATCC BAA-1605 (-) - CGTPACCIMRA':  '$\\bf{\\it{A. baumannii}}$ $\\bf{BAA\\text{-}1605}$',
    'E. cloacae ATCC 13047 (-)':                     '$\\it{E. cloacae}$ ATCC 13047',
    'E. coli ATCC 11775 (-)':                        '$\\it{E. coli}$ ATCC 11775',
    'E. coli AIC221 (-)':                            '$\\it{E. coli}$ AIC221',
    'E. coli AIC222 - CRE (-)':                      '$\\bf{\\it{E. coli}}$ $\\bf{AIC222}$',
    'E. coli ATCC BAA-3170 (-) - CRE':               '$\\bf{\\it{E. coli}}$ $\\bf{BAA\\text{-}3170}$',
    'K. pneumoniae ATCC 13883 (-)':                  '$\\it{K. pneumoniae}$ ATCC 13883',
    'K. pneumoniae ATCC BAA-2342 (-) - EIRK':        '$\\bf{\\it{K. pneumoniae}}$ $\\bf{BAA\\text{-}2342}$',
    'P. aeruginosa PAO1 (-)':                        '$\\it{P. aeruginosa}$ PAO1',
    'P. aeruginosa PA14 (-)':                        '$\\it{P. aeruginosa}$ PA14',
    'P. aeruginosa ATCC BAA-3197 (-) - FBCRP':       '$\\bf{\\it{P. aeruginosa}}$ $\\bf{BAA\\text{-}3197}$',
    'S. enterica ATCC 9150 (-)':                     '$\\it{S. enterica}$ ATCC 9150',
    'S. enterica Typhimurtium ATCC 700720':           '$\\it{S. enterica}$ Typhimurium',
    'B. subtilis ATCC 23857 (+)':                    '$\\it{B. subtilis}$ ATCC 23857',
    'S. aureus ATCC 12600 (+)':                      '$\\it{S. aureus}$ ATCC 12600',
    'S. aureus ATCC BAA-1556 - MRSA (+)':            '$\\bf{\\it{S. aureus}}$ $\\bf{BAA\\text{-}1556}$',
    'E. faecalis ATCC 700802 - VRE (+)':             '$\\bf{\\it{E. faecalis}}$ $\\bf{700802}$',
    'E. faecium ATCC 700221 - VRE (+)':              '$\\bf{\\it{E. faecium}}$ $\\bf{700221}$',
    'E. coli K-12 BW25113 (-)':                      '$\\it{E. coli}$ K-12 BW25113',
}
STRAIN_MDR = {s: any(t in s for t in ['CRAB', 'CRE', 'ESBL', 'EIRK', 'FBCR', 'FQR', 'MRSA', 'VRE', 'CGTPA'])
              for s in mic_strains}
STRAIN_GRAM = {s: ('+' if '(+)' in s else '-') for s in mic_strains}

print(f"Antimicrobial analogs: {len(antimicrobial)}")

Antimicrobial analogs: 59


In [4]:
# === Edit distance + property deltas: all analog families vs their prototype ===

def levenshtein(s1, s2):
    m, n = len(s1), len(s2)
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev, dp[0] = dp[0], i
        for j in range(1, n + 1):
            prev, dp[j] = dp[j], prev if s1[i-1]==s2[j-1] else 1+min(prev, dp[j], dp[j-1])
    return dp[n]

def hamming(s1, s2):
    return sum(a != b for a, b in zip(s1, s2)) if len(s1) == len(s2) else None

ALL_FAMILIES = sorted(ref[ref['category']=='analog']['prototype_display'].dropna().unique())

rows = []
for fam in ALL_FAMILIES:
    proto = ref[ref['short_name'] == fam]
    if proto.empty: continue
    p = proto.iloc[0]
    proto_seq = p['sequence'].upper().strip()
    analogs = ref[(ref['category'] == 'analog') & (ref['prototype_display'] == fam)]
    for _, row in analogs.iterrows():
        ana = row['sequence'].upper().strip()
        lev = levenshtein(proto_seq, ana)
        ham = hamming(proto_seq, ana)
        L   = max(len(proto_seq), len(ana))
        rows.append({
            'Family'      : fam,
            'Analog'      : row['short_name'],
            'Proto_len'   : int(p['length']),
            'Analog_len'  : len(ana),
            'Levenshtein' : lev,
            'Hamming'     : ham,
            'Pct_identity': round(100 * (1 - lev / L), 1),
            'dL'          : len(ana) - int(p['length']),
            'dC'          : round(row['net_charge']    - p['net_charge'],        2),
            'dH'          : round(row['mean_hydrophobicity'] - p['mean_hydrophobicity'], 3),
        })

edit_df = pd.DataFrame(rows)

summary = edit_df.groupby('Family').agg(
    Proto_len    =('Proto_len',    'first'),
    N            =('Analog',       'count'),
    Lev_med      =('Levenshtein',  'median'),
    Lev_range    =('Levenshtein',  lambda x: f'{x.min()}-{x.max()}'),
    PctID_med    =('Pct_identity', 'median'),
    PctID_range  =('Pct_identity', lambda x: f'{x.min():.1f}-{x.max():.1f}'),
    dL_med       =('dL',           'median'),
    dC_med       =('dC',           'median'),
    dH_med       =('dH',           'median'),
    All_subst    =('Hamming',      lambda x: (x == edit_df.loc[x.index,'Levenshtein']).all()),
).reset_index()
for col in ['Lev_med','PctID_med','dL_med','dC_med','dH_med']:
    summary[col] = summary[col].round(2)

print('=== Summary: edit distance + property deltas (median across analogs) ===\n')
print(summary[['Family','Proto_len','N','Lev_med','Lev_range',
               'PctID_med','PctID_range','dL_med','dC_med','dH_med','All_subst'
               ]].to_string(index=False))
print()
print('dL/dC/dH = analog minus prototype (length, net charge, mean Eisenberg hydrophobicity)')
print('All_subst = True: Levenshtein == Hamming for all analogs (pure substitutions)')
print()

print('=== Per-analog detail ===')
for fam in ALL_FAMILIES:
    sub = edit_df[edit_df['Family'] == fam]
    if sub.empty: continue
    print(f'\n--- {fam}  (prototype: {sub["Proto_len"].iloc[0]} AA) ---')
    print(sub[['Analog','Analog_len','Levenshtein','Hamming','Pct_identity',
               'dL','dC','dH']].to_string(index=False))


=== Summary: edit distance + property deltas (median across analogs) ===

     Family  Proto_len  N  Lev_med Lev_range  PctID_med PctID_range  dL_med  dC_med  dH_med  All_subst
As-CATH4-6L         17  9      8.0       7-9      52.90   47.1-58.8     0.0    4.99   -0.70      False
      BoCo1         22 10      6.5      5-11      70.45   50.0-77.3     0.0    0.00   -0.12      False
   DeNo1047         13 10      5.0       5-6      61.50   53.8-61.5     0.0    1.00   -0.13       True
       GQ20         20 10      8.0       6-9      60.00   55.0-70.0     0.0    2.95    0.15       True
       LG21         21 10      2.0       1-3      90.50   85.7-95.2     0.0    0.00    0.00       True
 Mammutin-1         15 10      6.0       4-6      60.00   60.0-73.3     0.0    2.96   -0.42      False
OP-145-TII4         24 10      8.5       5-9      64.60   62.5-79.2     0.0    0.00   -0.13       True
       bZIP         30 10     21.0     18-22      30.00   26.7-40.0     0.0    7.96    0.07      False

In [5]:
fig=plt.figure(figsize=(6.0,8.0),dpi=300)
gs_a   =gridspec.GridSpec(1,1,figure=fig,left=0.08,right=0.38,top=0.985,bottom=0.725)
gs_b   =gridspec.GridSpec(1,1,figure=fig,left=0.55,right=0.99,top=0.985,bottom=0.725)
# Panel C: 9-column grid with thin gap columns (col 2, col 5, col 7) to create
# subtle visual separation between 4 family groups:
#   group 1: Mammutin-1 (col 0) + GQ20 (col 1)
#   group 2: As-CATH4-6L (col 3) + BoCo1 (col 4)
#   group 3: OP-145-TII4 (col 6)
#   group 4: DeNo1047 (col 8)
gs_c   =gridspec.GridSpec(1,9,figure=fig,left=0.05,right=0.90,top=0.655,bottom=0.490,
                          wspace=0.10,width_ratios=[1,1,0.3,1,1,0.3,1,0.3,1])
# D/E and F/G: right=0.95 leaves a small margin so the right spine/ticks are not clipped
gs_de  =gridspec.GridSpec(1,2,figure=fig,left=0.13,right=0.95,top=0.445,bottom=0.260,wspace=0.45)
gs_fg  =gridspec.GridSpec(1,2,figure=fig,left=0.13,right=0.95,top=0.185,bottom=0.000,wspace=0.45)
L=dict(fontsize=8,fontweight='bold',va='top',ha='left')
sw_norm=TwoSlopeNorm(vmin=0.1,vcenter=1.0,vmax=128)
# Matches Figure S4B HC50 colormap: red < 1, white at 1, green > 1
sw_cmap=LinearSegmentedColormap.from_list('tox_gr', [
    (0.0, '#b2182b'), (0.2, '#ef8a62'), (0.4, '#fddbc7'),
    (0.5, '#f7f7f7'), (0.6, '#a6d96a'), (0.8, '#1a9641'),
    (1.0, '#006837'),
])

<Figure size 1800x2400 with 0 Axes>

In [6]:
# ────────────────────── A: strains MIC <= 2 dot-strip ──────────────────────
ax_a=fig.add_subplot(gs_a[0]); ax_a.text(-0.22,1.08,'A',transform=ax_a.transAxes,**L)
for i,fam in enumerate(INACTIVE_ORDER):
    proto=proto_by_fam.get(fam); ana=antimicrobial[antimicrobial['family']==fam]; color=FC[fam]
    p_str=proto['n_strains_le2'] if proto is not None and pd.notna(proto['n_strains_le2']) else 0
    ax_a.scatter(p_str,i,marker='D',c=color,edgecolors='none',s=18,zorder=5)
    if len(ana)>0:
        jitter=np.random.RandomState(42+i).normal(0,0.10,len(ana))
        for j,(idx,row) in enumerate(ana.iterrows()):
            if row['short_name'] in LEADS_SET:
                ax_a.scatter(row['n_strains_le2'],i+jitter[j],c=color,edgecolors='black',linewidth=0.5,s=14,marker='o',zorder=5)
                annotate_lead(ax_a,row['n_strains_le2'],i+jitter[j],LEADS_SHORT[row['short_name']],(8,7))
            else:
                ax_a.scatter(row['n_strains_le2'],i+jitter[j],c=color,edgecolors='none',linewidth=0.2,s=10,marker='o',alpha=0.9,zorder=4)
ax_a.set_xlabel('No. strains active (MIC $\\leq$ 2 $\\mu$M)',fontsize=6)
active_by_fam={}
for fam in INACTIVE_ORDER:
    ana_f=antimicrobial[antimicrobial['family']==fam]
    if len(ana_f)>0:
        active_by_fam[fam]=(int((ana_f['n_strains_le2']>=1).sum()),len(ana_f))
    else:
        active_by_fam[fam]=(0,0)
ylabels_text=[f"{f} ({active_by_fam[f][0]}/{active_by_fam[f][1]})" for f in INACTIVE_ORDER]
ax_a.set_yticks(range(len(INACTIVE_ORDER))); ylabels=ax_a.set_yticklabels(ylabels_text,fontsize=6)
for yl,fam in zip(ylabels,INACTIVE_ORDER): yl.set_color(FC[fam])
ax_a.invert_yaxis(); ax_a.set_xlim(-0.5,14.5)

_a_pos = ax_a.get_position()
_a_centre_x = (_a_pos.x0 + _a_pos.x1) / 2
fig.legend(
    handles=[
        Line2D([], [], marker='D', color='w', markerfacecolor='#555', markeredgecolor='none', markersize=5, lw=0, label='Prototype'),
        Line2D([], [], marker='o', color='w', markerfacecolor='#555', markeredgecolor='none', markersize=4, lw=0, label='Analog'),
    ],
    loc='lower center', bbox_to_anchor=(_a_centre_x, _a_pos.y1),
    bbox_transform=fig.transFigure, ncol=2, fontsize=6,
    frameon=True, framealpha=0.95, edgecolor='#ddd',
    handletextpad=0.3, columnspacing=0.8,
)

In [7]:
# ── Panel B: per-strain success rate ─────────────────────────────────────────
mic_ana = mic[mic['short_name'].isin(antimicrobial['short_name'])].copy()
mic_vals_ana = mic_ana[mic_strains].astype(float)

strain_data_b = []
for s in mic_strains:
    vals = mic_vals_ana[s]
    n_tested = vals.count()  # denominator = analogs actually tested against this strain
    strain_data_b.append({
        'name':     STRAIN_LABELS.get(s, s[:25]),
        'potent':   (vals <= 2).sum()  / n_tested * 100 if n_tested > 0 else 0,
        'active':   (vals <= 4).sum()  / n_tested * 100 if n_tested > 0 else 0,
        'moderate': (vals <= 32).sum() / n_tested * 100 if n_tested > 0 else 0,
        'mdr':      STRAIN_MDR[s],
        'gram':     STRAIN_GRAM[s],
    })

strain_df = pd.DataFrame(strain_data_b).sort_values('potent', ascending=True).reset_index(drop=True)
gram_neg_df = strain_df[strain_df['gram'] == '-'].reset_index(drop=True)
gram_pos_df = strain_df[strain_df['gram'] == '+'].reset_index(drop=True)

gs_b_split = gridspec.GridSpecFromSubplotSpec(
    2, 1, subplot_spec=gs_b[0],
    height_ratios=[len(gram_neg_df), len(gram_pos_df)], hspace=0.08,
)

ax_neg_b = fig.add_subplot(gs_b_split[0])
ax_neg_b.text(-0.35, 1.08, 'B', transform=ax_neg_b.transAxes, **L)
_dplot_b(ax_neg_b, gram_neg_df, show_xlabel=False)
ax_neg_b.tick_params(axis='x', bottom=False, labelbottom=False)
ax_neg_b.spines['bottom'].set_visible(False)

ax_pos_b = fig.add_subplot(gs_b_split[1])
_dplot_b(ax_pos_b, gram_pos_df)

_neg_pos = ax_neg_b.get_position()
_b_centre_x = (_neg_pos.x0 + _neg_pos.x1) / 2
fig.legend(
    handles=ax_neg_b.get_legend_handles_labels()[0],
    labels=ax_neg_b.get_legend_handles_labels()[1],
    loc='lower center', bbox_to_anchor=(_b_centre_x, _neg_pos.y1),
    bbox_transform=fig.transFigure, ncol=3, fontsize=6,
    frameon=True, framealpha=0.95, edgecolor='#ddd',
    handletextpad=0.3, columnspacing=0.8,
)

In [8]:
# ── Panel C: property space coloured by safety window ────────────────────────
C_COLS = [0, 1, 3, 4, 6, 8]

# De novo peptides shown as reference KDE backdrop in every sub-panel
denovo_ref = df[df['category'] == 'de_novo'].dropna(subset=['net_charge', 'mean_hydrophobicity'])

for i, fam in enumerate(INACTIVE_ORDER):
    ax = fig.add_subplot(gs_c[C_COLS[i]])
    if i == 0:
        ax.text(-0.25, 1.10, 'C', transform=ax.transAxes, **L)
    if len(denovo_ref) >= 10:
        sns.kdeplot(data=denovo_ref, x='net_charge', y='mean_hydrophobicity',
                    levels=5, color='gray', linewidths=0.4, alpha=0.5, ax=ax)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ana = antimicrobial[antimicrobial['family'] == fam].dropna(subset=['SW'])
    if len(ana) > 0:
        non_lead_analogs = ana[~ana['short_name'].isin(LEADS_SET)]
        if len(non_lead_analogs) > 0:
            ax.scatter(non_lead_analogs['net_charge'], non_lead_analogs['mean_hydrophobicity'],
                       c=non_lead_analogs['SW'].clip(0.1, 128), cmap=sw_cmap, norm=sw_norm,
                       marker='o', s=14, edgecolors='#777', linewidths=0.3, alpha=0.85, zorder=3)
        for _, row in ana[ana['short_name'].isin(LEADS_SET)].iterrows():
            ax.scatter(row['net_charge'], row['mean_hydrophobicity'],
                       c=[min(128, max(0.1, row['SW']))], cmap=sw_cmap, norm=sw_norm,
                       marker='o', s=18, edgecolors='black', linewidth=0.7, zorder=5)
            annotate_lead(ax, row['net_charge'], row['mean_hydrophobicity'],
                          LEADS_SHORT[row['short_name']], (-7, 7), 'right')
    proto = proto_by_fam.get(fam)
    if proto is not None:
        psw = proto.get('SW', np.nan)
        if pd.notna(psw):
            ax.scatter(proto['net_charge'], proto['mean_hydrophobicity'],
                       c=[min(128, max(0.1, psw))], cmap=sw_cmap, norm=sw_norm,
                       marker='D', edgecolors='#777', linewidths=0.3, s=15, zorder=5)
        else:
            ax.scatter(proto['net_charge'], proto['mean_hydrophobicity'],
                       marker='D', c=FC[fam], edgecolors='none', s=15, zorder=5)
    ax.set_title(fam, fontsize=6, color=FC[fam], pad=2)
    ax.set_xlim(-0.5, 10)
    ax.set_ylim(-0.85, 0.55)
    if i == 0:
        ax.set_ylabel('Hydrophobicity', fontsize=6)
    else:
        ax.set_yticklabels([])

fig.text(0.475, 0.465, 'Net charge', ha='center', va='top', fontsize=6)
cbar_ax = fig.add_axes([0.92, 0.490, 0.012, 0.165])
cb = plt.colorbar(ScalarMappable(norm=sw_norm, cmap=sw_cmap), cax=cbar_ax)
cb.set_label('Safety Window\n(HC$_{50}$/MIC$_{50}$)', fontsize=6)
cb.set_ticks([0.1, 1, 10, 128])
cb.set_ticklabels(['0.1', '1', '10', '$\\geq$128'])
cb.ax.tick_params(labelsize=6)

<Figure size 1920x1320 with 0 Axes>

In [9]:
# ────────── D, E: HC50 and CC50 by family (NEW) ─────────────────────────────
LEAD_OFFSETS_DE={
    'D':{'Ω-AP-Mammutin-1-4':(-12,15),'Ω-AT-BoCo1-9':(-12,9),
         'Ω-AT-BoCo1-5':(-12,-9),'Ω-AU-GQ20-4':(-12,-15)},
    'E':{'Ω-AP-Mammutin-1-4':(10,10),'Ω-AT-BoCo1-5':(-10,10),
         'Ω-AT-BoCo1-9':(10,-10),'Ω-AU-GQ20-4':(10,0)},
}

def _safety_panel(ax,metric,xlabel_text,panel_key):
    cap=128.0; floor=0.1

    ax.axvline(cap,color='#bbbbbb',ls=':',lw=0.5,zorder=1)

    dn_vals=df[df['category']=='de_novo'][[metric,f'{metric}_raw']].dropna(subset=[metric])
    if len(dn_vals)>0:
        dn_jit=np.random.RandomState(7).normal(0,0.10,len(dn_vals))
        for j,(_,r) in enumerate(dn_vals.iterrows()):
            raw=r[f'{metric}_raw']; v=r[metric]
            censored=pd.notna(raw) and raw>cap
            if censored:
                ax.scatter(v,-1+dn_jit[j],facecolors='white',edgecolors='#888888',linewidth=0.4,s=8,alpha=0.7,zorder=3)
            else:
                ax.scatter(v,-1+dn_jit[j],c='#888888',edgecolors='none',s=7,alpha=0.5,zorder=3)

    for i,fam in enumerate(INACTIVE_ORDER):
        color=FC[fam]
        proto=proto_by_fam.get(fam)
        ana=antimicrobial[antimicrobial['family']==fam].copy()

        if proto is not None and pd.notna(proto[metric]):
            raw=proto[f'{metric}_raw']
            v=min(max(raw,floor),cap)
            if pd.notna(raw) and raw>cap:
                ax.scatter(v,i,marker='D',facecolors='white',edgecolors=color,linewidth=0.9,s=22,zorder=6)
            else:
                ax.scatter(v,i,marker='D',c=color,edgecolors='none',s=22,zorder=5)

        if len(ana)==0: continue
        jit=np.random.RandomState(11+i+(0 if metric=='HC50' else 100)).normal(0,0.12,len(ana))
        for j,(idx,row) in enumerate(ana.iterrows()):
            raw=row[f'{metric}_raw']; v=row[metric]
            is_lead=row['short_name'] in LEADS_SET
            censored_high = pd.notna(raw) and raw>cap
            if is_lead:
                if censored_high:
                    ax.scatter(v,i+jit[j],facecolors='white',edgecolors='black',linewidth=0.7,s=18,zorder=6)
                else:
                    ax.scatter(v,i+jit[j],c=color,edgecolors='black',linewidth=0.5,s=14,zorder=6)
                off=LEAD_OFFSETS_DE[panel_key].get(row['short_name'],(10,0))
                ha='right' if off[0]<0 else 'left'
                annotate_lead(ax,v,i+jit[j],LEADS_SHORT[row['short_name']],off,ha=ha)
            else:
                if censored_high:
                    ax.scatter(v,i+jit[j],facecolors='white',edgecolors=color,linewidth=0.6,s=11,alpha=0.95,zorder=3)
                else:
                    ax.scatter(v,i+jit[j],c=color,edgecolors='none',s=10,marker='o',alpha=0.85,zorder=3)

    groups=[]
    for fam in INACTIVE_ORDER:
        fd=antimicrobial[antimicrobial['family']==fam][metric].dropna()
        if len(fd)>0: groups.append(fd.values)
    H,p=kruskal(*groups)
    n_total=sum(len(g) for g in groups); k=len(groups)
    eta2 = H / (n_total - 1)

    ax.set_xscale('log')
    ax.set_xlim(0.07,160)
    ax.set_xticks([0.1,1,10,100])
    ax.set_xticklabels(['0.1','1','10','100'])
    ax.set_yticks([-1]+list(range(len(INACTIVE_ORDER))))
    ylabs=ax.set_yticklabels(['$\\it{De\\;novo}$']+INACTIVE_ORDER,fontsize=6)
    ylabs[0].set_color('#666')
    for yl,fam in zip(ylabs[1:],INACTIVE_ORDER): yl.set_color(FC[fam])
    ax.set_ylim(5.7,-1.7)
    ax.set_xlabel(xlabel_text,fontsize=6)
    ax.text(0.03,1.02,f'KW $p$ = {p:.1e}, $\\eta^2$ = {eta2:.2f}',transform=ax.transAxes,
            fontsize=6,ha='left',va='bottom',color='#555',fontweight='bold',clip_on=False)

ax_d=fig.add_subplot(gs_de[0]); ax_d.text(-0.20,1.08,'D',transform=ax_d.transAxes,**L)
_safety_panel(ax_d,'HC50','HC$_{50}$ ($\\mu$M)','D')
ax_e=fig.add_subplot(gs_de[1]); ax_e.text(-0.20,1.08,'E',transform=ax_e.transAxes,**L)
_safety_panel(ax_e,'CC50','CC$_{50}$ ($\\mu$M)','E')

In [10]:
# ── Panels F, G: NPN outer-membrane permeabilization / DiSC membrane depolarization ──
LEAD_OFFSETS_FG = {
    'Ω-AT-BoCo1-5':       {'NPN': (10, -14), 'DiSC': (10, -12)},
    'Ω-AT-BoCo1-9':       {'NPN': (10,  14), 'DiSC': (10,  12)},
    'Ω-AU-GQ20-4':        {'NPN': (10, -12), 'DiSC': (10, -12)},
    'Ω-AP-Mammutin-1-4':  {'NPN': (10,  12), 'DiSC': (10,  12)},
}

npn_med  = df.dropna(subset=['NPN'])['NPN'].median()
disc_med = df.dropna(subset=['DiSC'])['DiSC'].median()

# Separate subsets per assay — NPN and DiSC should be co-measured, but use
# independent dropna so a peptide missing one assay is not excluded from both panels.
mech_npn  = antimicrobial.dropna(subset=['NPN']).copy()
mech_disc = antimicrobial.dropna(subset=['DiSC']).copy()

# F/G use reversed family order so the top family in A–E appears at the top here too
fam_order_fg = INACTIVE_ORDER[::-1]

for pi, (metric, mech_df, med_val, xlabel_text, letter) in enumerate([
    ('NPN',  mech_npn,  npn_med,
     'Outer membrane permeabilization\nNPN MaxRel (%) at FC', 'F'),
    ('DiSC', mech_disc, disc_med,
     'Cytoplasmic membrane depolarization\nDiSC$_3$(5) MaxRel (%) at FC', 'G'),
]):
    ax = fig.add_subplot(gs_fg[pi])
    ax.text(-0.20, 1.08, letter, transform=ax.transAxes, **L)

    for i, fam in enumerate(fam_order_fg):
        fam_vals = mech_df[mech_df['family'] == fam]
        color = FC[fam]
        if len(fam_vals) == 0:
            continue
        proto = proto_by_fam.get(fam)
        if proto is not None:
            proto_rows = df[df['short_name'] == proto['short_name']].dropna(subset=[metric])
            if len(proto_rows) > 0:
                ax.scatter(proto_rows[metric].values, [i] * len(proto_rows),
                           marker='D', c=color, edgecolors='none', s=15, zorder=5)
        jit = np.random.RandomState(i + 300 + pi).normal(0, 0.10, len(fam_vals))
        for j, (_, row) in enumerate(fam_vals.iterrows()):
            if row['short_name'] in LEADS_SET:
                ax.scatter(row[metric], i + jit[j],
                           c=color, s=14, edgecolors='black', linewidth=0.5, zorder=5)
                off = LEAD_OFFSETS_FG.get(row['short_name'], {}).get(metric, (10, 10))
                annotate_lead(ax, row[metric], i + jit[j], LEADS_SHORT[row['short_name']], off)
            else:
                ax.scatter(row[metric], i + jit[j],
                           c=color, s=12, marker='o', edgecolors='none',
                           linewidth=0.2, alpha=0.7, zorder=3)

    ax.axvline(med_val, color='#ccc', ls=':', lw=0.5, zorder=1)
    ax.set_yticks(list(range(len(fam_order_fg))))
    fam_labels = [f"{f}  ({FAM_CONC[f]:g} μM)" if f in FAM_CONC else f for f in fam_order_fg]
    yl = ax.set_yticklabels(fam_labels, fontsize=6)
    for y, f in zip(yl, fam_order_fg):
        y.set_color(FC[f])
    ax.set_xlabel(xlabel_text, fontsize=6)

    groups = [mech_df[mech_df['family'] == f][metric].dropna().values
              for f in INACTIVE_ORDER if len(mech_df[mech_df['family'] == f]) > 0]
    groups = [g for g in groups if len(g) > 0]
    if len(groups) >= 2:
        H, p = kruskal(*groups)
        n_total = sum(len(g) for g in groups)
        k = len(groups)
        # Epsilon-squared effect size: Schaich & Anderson (1994) variant
        eta2 = H / (n_total - 1)
        if p > 0.05:
            stats_txt = f'KW $p$ > 0.05, $\\eta^2$ = {eta2:.2f}'
        else:
            stats_txt = f'KW $p$ = {p:.1e}, $\\eta^2$ = {eta2:.2f}'
        ax.text(0.03, 1.02, stats_txt,
                transform=ax.transAxes, fontsize=6, ha='left', va='bottom',
                color='#555', fontweight='bold', clip_on=False)

In [11]:
OUT = '/home/pszymczak/code/omegamp-dashboard/figures/'
fig.savefig(OUT + 'figure5_analog.pdf', bbox_inches='tight')
fig.savefig(OUT + 'figure5_analog.svg', bbox_inches='tight')
fig.savefig(OUT + 'figure5_analog.png', dpi=400, bbox_inches='tight')
plt.show()